In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tabml.datasets

In [2]:
df = tabml.datasets.download_movielen_1m()
df.keys()

dict_keys(['users', 'movies', 'ratings'])

In [3]:
users, ratings, movies = df['users'], df['ratings'], df['movies']
print('Number of user: ', len(users['UserID']))
print('Number of movie: ', len(movies['MovieID']))

Number of user:  6040
Number of movie:  3883


In [4]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   UserID      6040 non-null   int64 
 1   Gender      6040 non-null   object
 2   Age         6040 non-null   int64 
 3   Occupation  6040 non-null   int64 
 4   Zip-code    6040 non-null   object
dtypes: int64(3), object(2)
memory usage: 236.1+ KB


In [5]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   MovieID  3883 non-null   int64 
 1   Title    3883 non-null   object
 2   Genres   3883 non-null   object
dtypes: int64(1), object(2)
memory usage: 91.1+ KB


In [6]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   UserID     1000209 non-null  int64
 1   MovieID    1000209 non-null  int64
 2   Rating     1000209 non-null  int64
 3   Timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


In [7]:
# CREATING UTILITY MATRIX: USER-ITEM, ITEM-RATING.

In [8]:
# user_index_by_id = {id: idx for id,idx in enumerate(users['UserID'])}
# movie_index_by_id = {id: idx for id, idx in enumerate(movies['MovieID'])}


# print(len(user_index_by_id))
# print(len(movie_index_by_id))

In [9]:
genre_list = []
for genres in movies['Genres'].unique():
    for genre in genres.split('|'):
        if  genre not in genre_list:
            genre_list.append(genre)

genre_list

['Animation',
 "Children's",
 'Comedy',
 'Adventure',
 'Fantasy',
 'Romance',
 'Drama',
 'Action',
 'Crime',
 'Thriller',
 'Horror',
 'Sci-Fi',
 'Documentary',
 'War',
 'Musical',
 'Mystery',
 'Film-Noir',
 'Western']

In [10]:
genre_index_by_name = {name:i for i, name in enumerate(genre_list)}
movie_features = np.zeros((len(movies), len(genre_list)))

for idx, movie_genres in enumerate(movies['Genres']):
    for genre in movie_genres.split('|'):
        genre_idx = genre_index_by_name[genre]
        movie_features[idx, genre_idx] = 1

movie_features

array([[1., 1., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [11]:
from tqdm import tqdm

movie_index_by_id = [movie for movie in movies['MovieID']]
user_index_by_id = [user for user in users['UserID']]

user_item = np.zeros((len(user_index_by_id), len(movie_index_by_id)))

for user_id in tqdm(ratings['UserID'].unique()):
    for movie_id in ratings[ratings['UserID'] == user_id]['MovieID']:
        movie_idx = movie_index_by_id.index(movie_id)
        user_idx = user_index_by_id.index(user_id)
        user_item[user_idx, movie_idx] = ratings[ratings['UserID'] == user_id][ratings['MovieID'] == movie_id]['Rating']
user_item

  0%|          | 0/6040 [00:00<?, ?it/s]/tmp/ipykernel_231590/3901769014.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  user_item[user_idx, movie_idx] = ratings[ratings['UserID'] == user_id][ratings['MovieID'] == movie_id]['Rating']
100%|██████████| 6040/6040 [53:28<00:00,  1.88it/s]  


array([[5., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [3., 0., 0., ..., 0., 0., 0.]])

In [26]:
# Similarity matrix: similarity between users on a specific item.

def cosine(vect1, vect2):
    numerator = np.dot(vect1, vect2)
    denominator = np.linalg.norm(vect1)*np.linalg.norm(vect2)
    return numerator/denominator

def pearson(vect1, vect2):
    return np.corrcoef(vect1, vect2)[0, 1]

def similarity(vect1, vect2, type='euclidean'):
    if type==None or type == 'euclidean':
        return cosine(vect1, vect2)
    if type == 'pearson':
        return pearson(vect1, vect2)
    else:
        print('There is that type of built-in similarity function')



In [27]:
df_example = user_item[0: 100, :]
df_example

array([[5., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [3., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [40]:
N = df_example.shape[0]
similarity_matrix = np.zeros((N, N))

for row in tqdm(range(similarity_matrix.shape[0])):
    active_user = df_example[row, :]
    for col in range(similarity_matrix.shape[1]):
        peer = df_example[col, : ]
        similarity_matrix[row, col] = similarity(active_user, peer)

similarity_matrix.shape

100%|██████████| 100/100 [00:00<00:00, 573.82it/s]


(100, 100)

In [78]:
def neighbor(sim_mat, active_user, k_neighbors):
    sorted_matrix = sorted(sim_mat[active_user, :], reverse=True)
    chosen = sorted_matrix[: k_neighbors + 1]
    idx_chosen = [np.where(sim_mat[active_user, :] == number)[0][0] for number in chosen]
    return chosen, idx_chosen

neighbor(similarity_matrix, 10, 20)

([1.0,
  0.2934483865402258,
  0.2920190407847125,
  0.27668191635089806,
  0.2722487383131304,
  0.2628202917171918,
  0.26087575682536013,
  0.2535497882127949,
  0.2523904207231588,
  0.2452594634030471,
  0.2451287185684664,
  0.242910435592721,
  0.24024405205881538,
  0.2397669370232238,
  0.23657442641031667,
  0.2298251595674826,
  0.22860756929573953,
  0.22804443755023793,
  0.22626063054382073,
  0.22265008929804372,
  0.22097778947826593],
 [10,
  8,
  21,
  55,
  23,
  75,
  14,
  89,
  27,
  57,
  94,
  47,
  81,
  33,
  79,
  71,
  91,
  9,
  54,
  95,
  4])

In [91]:
def prediction(user_item, sim_mat, k_neighbors, active_user):
    mean_rate_active_user = np.mean(user_item[active_user, : ])
    chosen_rate, chosen_idx = neighbor(sim_mat, active_user, k_neighbors)

    numerator = 0
    denominator = np.sum(chosen_rate)

    for k_th in range(k_neighbors):
        mean_centered_rating_items = user_item[chosen_idx[k_th], :] - np.mean(user_item[chosen_idx[k_th], : ])
        numerator += chosen_rate[k_th] * mean_centered_rating_items

    return mean_rate_active_user + (numerator/denominator)

neigbors, idx = neighbor(similarity_matrix, 1, 10)

np.where(prediction(df_example, similarity_matrix, 10, 10) > 0.8)

(array([   0,   24,   31,   33,   35,   38,   46,   49,   87,  102,  108,
         214,  220,  228,  243,  257,  290,  293,  297,  315,  329,  338,
         341,  352,  360,  363,  436,  437,  450,  453,  470,  476,  477,
         523,  539,  547,  583,  586,  589,  593,  604,  642,  657,  711,
         754,  768,  774,  778,  847,  890,  900,  907,  908,  982, 1045,
        1058, 1064, 1073, 1120, 1171, 1176, 1178, 1179, 1180, 1182, 1192,
        1195, 1196, 1203, 1207, 1211, 1220, 1224, 1227, 1229, 1239, 1245,
        1250, 1258, 1268, 1271, 1287, 1337, 1356, 1370, 1373, 1385, 1388,
        1416, 1439, 1468, 1491, 1507, 1523, 1539, 1543, 1575, 1627, 1636,
        1653, 1656, 1672, 1683, 1695, 1696, 1700, 1720, 1726, 1840, 1847,
        1852, 1854, 1899, 1959, 1995, 2007, 2037, 2040, 2105, 2213, 2233,
        2252, 2255, 2260, 2266, 2286, 2290, 2326, 2327, 2470, 2473, 2502,
        2511, 2559, 2588, 2614, 2623, 2637, 2647, 2677, 2693, 2697, 2722,
        2726, 2735, 2757, 2789, 2821, 

In [127]:
predict = list(np.where(prediction(df_example, similarity_matrix, 10, 10) > 0.8)[0])

df_predict = pd.DataFrame()
df_predict.columns = movies.columns
# df_predict.loc[len(df_predict.index)] = list(np.array(movies[movies['MovieID'] == 1])[0])
df_predict

ValueError: Length mismatch: Expected axis has 0 elements, new values have 3 elements

In [132]:
print(list(np.array(movies[movies['MovieID'] == 1])[0]))
# print(movies[movies['MovieID'] == 1])

[1, 'Toy Story (1995)', "Animation|Children's|Comedy"]


In [154]:
predict = list(np.where(prediction(df_example, similarity_matrix, 10, 20) > 0.8)[0])
# predict
data_ = []
for item in predict: 
    movie = np.array(movies[movies['MovieID'] == (item + 1)])
    if movie is not None: 
        data_.append(movie)

for data in data_:
    print(list(data))

[array([1, 'Toy Story (1995)', "Animation|Children's|Comedy"], dtype=object)]
[array([34, 'Babe (1995)', "Children's|Comedy|Drama"], dtype=object)]
[array([109, 'Headless Body in Topless Bar (1995)', 'Comedy'], dtype=object)]
[array([197, 'Stars Fell on Henrietta, The (1995)', 'Drama'], dtype=object)]
[]
[array([258, "Kid in King Arthur's Court, A (1995)",
       "Adventure|Children's|Comedy|Fantasy|Romance"], dtype=object)]
[array([294, 'Perez Family, The (1995)', 'Comedy|Romance'], dtype=object)]
[array([374, 'Richie Rich (1994)', "Children's|Comedy"], dtype=object)]
[array([377, 'Speed (1994)', 'Action|Romance|Thriller'], dtype=object)]
[array([454, 'Firm, The (1993)', 'Drama|Thriller'], dtype=object)]
[array([477, "What's Love Got to Do with It? (1993)", 'Drama'],
      dtype=object)]
[array([524, 'Rudy (1993)', 'Drama'], dtype=object)]
[array([548, 'Terminal Velocity (1994)', 'Action'], dtype=object)]
[array([582, 'Metisse (Café au Lait) (1993)', 'Comedy'], dtype=object)]
[array([